In [ ]:
# Parameters for Papermill / development
conversation_id = "dev-conversation"
collection_name = "conversation_dev"
file_paths = ["../sample-data/Introduction_to_Tableau.pdf"]



# LangChain Project - Parameterized Notebook

This notebook keeps the development notebook style, but accepts dynamic file inputs so it can be executed by the Node/Koa backend.

It reuses the same shared Python functions used by the production scripts.


In [ ]:

%load_ext dotenv
%dotenv

from shared.extractors import extract_many
from shared.chunkers import split_into_chunks
from shared.vector_store import upsert_chunks
from shared.suggested_questions import suggest_questions_from_chunks
from shared.rag import answer_with_citations


In [ ]:

documents = extract_many(file_paths)
documents


In [ ]:

all_chunks = []

for document in documents:
    chunks = split_into_chunks(document["file_name"], document["text"])
    all_chunks.extend(chunks)

len(all_chunks)


In [ ]:

result_upsert = upsert_chunks(
    collection_name=collection_name,
    conversation_id=conversation_id,
    chunks=all_chunks,
)

result_upsert


In [ ]:

suggested_questions = suggest_questions_from_chunks([chunk.text for chunk in all_chunks])
suggested_questions


In [ ]:

sample_answer = answer_with_citations(
    collection_name=collection_name,
    conversation_id=conversation_id,
    question=suggested_questions[0] if suggested_questions else "What is this document about?"
)

sample_answer


In [ ]:

import json

print(json.dumps({
    "conversation_id": conversation_id,
    "collection_name": collection_name,
    "chunk_count": len(all_chunks),
    "suggested_questions": suggested_questions,
}, ensure_ascii=False))
